# VibeTrader: Diffusion Model Chart Prediction

Fine-tune a diffusion model (InstructPix2Pix on SD 1.5) to predict financial chart continuations from images, then trade on the predictions.

**Paper reference**: [arXiv:2509.02308](https://arxiv.org/abs/2509.02308)

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

# Project imports
sys.path.insert(0, ".")
from data.render_charts import render_candlestick, compute_signal
from inference.extract_signal import extract_signal

%matplotlib inline
plt.style.use("dark_background")

## 2. Data Pipeline Demo

Show sample chart pairs from the training data.

In [ ]:
# Load OHLCV data
csv_path = "data/btc_usdt_4h.csv"
df = pd.read_csv(csv_path)
print(f"Total candles: {len(df)}")
df.head()

In [ ]:
# Render sample chart pairs
WINDOW = 40
FUTURE = 4

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle("Sample Chart Pairs: Input | Predicted Target | Signal", fontsize=16)

sample_indices = np.linspace(100, len(df) - WINDOW - FUTURE - 1, 3, dtype=int)

for row, idx in enumerate(sample_indices):
    input_window = df.iloc[idx : idx + WINDOW]
    target_window = df.iloc[idx + FUTURE : idx + FUTURE + WINDOW]
    
    current_close = input_window.iloc[-1]["close"]
    future_close = target_window.iloc[-1]["close"]
    signal, marker_color = compute_signal(current_close, future_close)
    pct = (future_close - current_close) / current_close
    
    input_img = render_candlestick(input_window)
    target_img = render_candlestick(target_window, draw_marker=True, marker_color=marker_color)
    
    axes[row, 0].imshow(input_img)
    axes[row, 0].set_title(f"Input (candle {idx})")
    axes[row, 0].axis("off")
    
    axes[row, 1].imshow(target_img)
    axes[row, 1].set_title(f"Target ({signal}, {pct:+.1%})")
    axes[row, 1].axis("off")
    
    # Show marker detail
    marker_detail = np.array(target_img)[221:251, 221:251]
    axes[row, 2].imshow(marker_detail, interpolation="nearest")
    axes[row, 2].set_title(f"Marker: {signal}")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()

## 3. Model Inference

Load a trained checkpoint and generate predictions.

In [ ]:
from inference.predict import load_pipeline, predict

# Update this path to your checkpoint
CHECKPOINT = "checkpoints"  # or "checkpoints/checkpoint-500" for an earlier checkpoint

pipe = load_pipeline(CHECKPOINT)
print("Model loaded!")

In [ ]:
# Run inference on sample charts
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle("Input | Model Prediction | Actual Target", fontsize=16)

# Use held-out test data (last 3 months)
df["timestamp"] = pd.to_datetime(df["timestamp"])
cutoff = df["timestamp"].max() - pd.DateOffset(months=3)
test_df = df[df["timestamp"] >= cutoff].reset_index(drop=True)
test_indices = np.linspace(0, len(test_df) - WINDOW - FUTURE - 1, 3, dtype=int)

for row, idx in enumerate(test_indices):
    input_window = test_df.iloc[idx : idx + WINDOW]
    target_window = test_df.iloc[idx + FUTURE : idx + FUTURE + WINDOW]
    
    current_close = input_window.iloc[-1]["close"]
    future_close = target_window.iloc[-1]["close"]
    actual_signal, marker_color = compute_signal(current_close, future_close)
    
    # Render input
    input_img = render_candlestick(input_window)
    
    # Build prompt
    rsi = input_window.iloc[-1].get("rsi", 50.0)
    macd = input_window.iloc[-1].get("MACD_12_26_9", 0.0)
    prompt = f"Predict next {FUTURE} candles. RSI={round(float(rsi), 1)}, MACD={round(float(macd), 2)}"
    
    # Model prediction
    generated = predict(pipe, input_img, prompt)
    pred_signal = extract_signal(generated)
    
    # Actual target
    actual_img = render_candlestick(target_window, draw_marker=True, marker_color=marker_color)
    
    pct = (future_close - current_close) / current_close
    
    axes[row, 0].imshow(input_img)
    axes[row, 0].set_title(f"Input")
    axes[row, 0].axis("off")
    
    axes[row, 1].imshow(generated)
    axes[row, 1].set_title(f"Predicted: {pred_signal.action} (conf={pred_signal.confidence:.2f})")
    axes[row, 1].axis("off")
    
    axes[row, 2].imshow(actual_img)
    axes[row, 2].set_title(f"Actual: {actual_signal} ({pct:+.1%})")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()

## 4. Signal Extraction Demo

Show how the colored marker is read to determine BUY/SELL/HOLD.

In [ ]:
from inference.extract_signal import extract_marker_region

# Create test images with known markers
test_signals = [
    ("BUY", (0, 255, 0)),
    ("SELL", (255, 0, 0)),
    ("HOLD", (128, 128, 128)),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for i, (label, color) in enumerate(test_signals):
    # Use a sample chart with the marker
    sample_window = df.iloc[100:140]
    img = render_candlestick(sample_window, draw_marker=True, marker_color=color)
    
    signal = extract_signal(img)
    marker = extract_marker_region(img)
    
    axes[i].imshow(img)
    axes[i].set_title(f"Expected: {label}\nDetected: {signal.action} (conf={signal.confidence:.2f})\nRGB: {signal.avg_rgb}")
    axes[i].axis("off")

plt.suptitle("Signal Extraction Verification", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Backtest Results

Load and visualize backtest metrics.

In [ ]:
import json

# Load backtest results
results_path = Path("outputs/backtest")

if (results_path / "backtest_metrics.json").exists():
    with open(results_path / "backtest_metrics.json") as f:
        metrics = json.load(f)
    
    print("Backtest Metrics:")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.2%}" if "return" in k or "accuracy" in k else f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
    
    # Load detailed results
    if (results_path / "backtest_results.csv").exists():
        bt = pd.read_csv(results_path / "backtest_results.csv")
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        # Signal distribution
        bt["predicted"].value_counts().plot(kind="bar", ax=axes[0], color=["green", "red", "gray"])
        axes[0].set_title("Predicted Signal Distribution")
        
        # Cumulative returns
        bt["cum_strategy"] = (1 + bt["strategy_return"]).cumprod()
        bt["cum_buyhold"] = (1 + bt["pct_change"]).cumprod()
        axes[1].plot(bt["cum_strategy"], label="Strategy", color="cyan")
        axes[1].plot(bt["cum_buyhold"], label="Buy & Hold", color="orange")
        axes[1].legend()
        axes[1].set_title("Cumulative Returns")
        
        # Accuracy by signal
        acc_by_signal = bt.groupby("predicted")["correct"].mean()
        acc_by_signal.plot(kind="bar", ax=axes[2], color=["green", "red", "gray"])
        axes[2].set_title("Accuracy by Signal")
        axes[2].axhline(y=1/3, color="white", linestyle="--", label="Random (33%)")
        axes[2].legend()
        
        plt.tight_layout()
        plt.show()
else:
    print("No backtest results found. Run backtest first:")
    print("  python bot/backtest.py --checkpoint checkpoints")

## 6. Quick Commands Reference

```bash
# 1. Fetch data
python data/fetch_ohlcv.py

# 2. Render chart pairs
python data/render_charts.py

# 3. Build HuggingFace dataset
python data/build_dataset.py

# 4. Train (launches in background)
bash train/run_training.sh

# 5. Run inference
python inference/predict.py --checkpoint checkpoints --input data/rendered/input

# 6. Backtest
python bot/backtest.py --checkpoint checkpoints

# 7. Paper trade (stretch goal)
python bot/trader.py --checkpoint checkpoints
```